# Animación 3D de 1I/ʻOumuamua cerca del Sol

Este notebook genera una animación 3D de la órbita de 1I/ʻOumuamua en las cercanías del Sol, mostrando también la Tierra. Se presentan dos versiones: una con **matplotlib** y otra con **plotly**.

In [ ]:
import numpy as np
import pandas as pd
import pymcel as pc
from pymcel import constantes as const
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import plotly.graph_objects as go

In [ ]:
# Constantes y utilidades

def _to_float(x):
    return float(x.value) if hasattr(x, 'value') else float(x)

# Preferimos mu_sun (SPICE, típicamente km^3/s^2); fallback a GM_sun.
MU_SUN_RAW = getattr(const, 'mu_sun', getattr(const, 'GM_sun'))
MU_SUN = _to_float(MU_SUN_RAW)

# Si viene en SI (m^3/s^2), convertir a km^3/s^2.
if MU_SUN > 1e15:
    MU_SUN = MU_SUN / 1e9

AU_KM = const.au.to('km').value if hasattr(const.au, 'to') else 149597870.7

print(f"mu_sun usado = {MU_SUN:.6e} km^3/s^2")
print(f"1 AU = {AU_KM:.6e} km")

In [ ]:
# 1) Estado inicial de Oumuamua (Horizons)

epoch = '2017-10-21'

resultado_obj = pc.consulta_horizons(
    id='1I',
    location='@0',
    epochs=epoch,
    datos='vectors',
    propiedades=[
        ('x', 'km'), ('y', 'km'), ('z', 'km'),
        ('vx', 'km/s'), ('vy', 'km/s'), ('vz', 'km/s')
    ]
)


def extrae_estado_km(resultado):
    # Horizons devuelve (tabla, epoch_jd, vector_raw).
    if isinstance(resultado, tuple) and len(resultado) >= 3:
        estado = np.asarray(resultado[2], dtype=float)
        if estado.ndim == 2:
            estado = estado[0]
        return estado

    # Fallback por si se pasa una tabla o DataFrame directamente.
    if hasattr(resultado, 'columns'):
        norm = {str(c).lower().replace('_', ''): c for c in resultado.columns}
        keys = ['x', 'y', 'z', 'vx', 'vy', 'vz']
        vals = []
        for k in keys:
            if k not in norm:
                raise KeyError(f"No se encontró columna '{k}' en {list(resultado.columns)}")
            vals.append(float(resultado.iloc[0][norm[k]]))
        return np.array(vals, dtype=float)

    raise TypeError(f"Tipo no soportado para extraer estado: {type(resultado)}")


estado = extrae_estado_km(resultado_obj)
r0 = estado[:3]
v0 = estado[3:]

epoch_jd = float(resultado_obj[1]) if isinstance(resultado_obj, tuple) and len(resultado_obj) >= 2 else float(
    pd.to_datetime(epoch).to_julian_date()
)

print('r0 [km]   =', r0)
print('v0 [km/s] =', v0)
print('epoch_jd  =', epoch_jd)

In [ ]:
# 2) Efemérides de la Tierra en el mismo intervalo temporal

window_days = 365
step_days = 2

start_date = (pd.to_datetime(epoch) - pd.Timedelta(days=window_days)).strftime('%Y-%m-%d')
stop_date = (pd.to_datetime(epoch) + pd.Timedelta(days=window_days)).strftime('%Y-%m-%d')

resultado_tierra = pc.consulta_horizons(
    id='399',
    location='@0',
    epochs={'start': start_date, 'stop': stop_date, 'step': f'{step_days}d'},
    datos='vectors',
    propiedades=[
        ('x', 'km'), ('y', 'km'), ('z', 'km'),
        ('vx', 'km/s'), ('vy', 'km/s'), ('vz', 'km/s')
    ]
)

tabla_tierra = resultado_tierra[0]
df_tierra = tabla_tierra.to_pandas() if hasattr(tabla_tierra, 'to_pandas') else pd.DataFrame(tabla_tierra)

norm_cols = {str(c).lower().replace('_', ''): c for c in df_tierra.columns}

def _col(name):
    key = name.lower().replace('_', '')
    if key not in norm_cols:
        raise KeyError(f"No se encontró columna '{name}' en {list(df_tierra.columns)}")
    return norm_cols[key]

x_col, y_col, z_col = _col('x'), _col('y'), _col('z')
ijd_col = _col('datetime_jd')

r_earth_km = df_tierra[[x_col, y_col, z_col]].to_numpy(dtype=float)

ts = (df_tierra[ijd_col].to_numpy(dtype=float) - epoch_jd) * 86400.0

In [ ]:
# 3) Integración de la órbita de Oumuamua con el mismo vector temporal

rs_obj, vs_obj = pc.doscuerpos_solucion(MU_SUN, r0, v0, ts)

rs_obj = np.asarray(rs_obj)
vs_obj = np.asarray(vs_obj)

if rs_obj.ndim == 3 and rs_obj.shape[0] == 1:
    rs_obj = rs_obj[0]
if vs_obj.ndim == 3 and vs_obj.shape[0] == 1:
    vs_obj = vs_obj[0]

if rs_obj.ndim != 2 or rs_obj.shape[1] != 3:
    raise ValueError(f"Forma inesperada de rs_obj: {rs_obj.shape}")

print(f"Integración completada: {len(ts)} pasos temporales")

In [ ]:
# 4) Conversión a AU para visualización

r_earth_au = r_earth_km / AU_KM
r_obj_au = rs_obj / AU_KM

## Animación 3D con Matplotlib

In [ ]:
frame_step = max(1, len(ts) // 200)
frame_indices = np.arange(0, len(ts), frame_step)

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')

sun = ax.scatter([0], [0], [0], color='gold', s=80, label='Sol')

earth_line, = ax.plot([], [], [], color='royalblue', linewidth=1, alpha=0.6, label='Órbita Tierra')
oumuamua_line, = ax.plot([], [], [], color='black', linewidth=1, alpha=0.7, label='Órbita Oumuamua')

earth_marker = ax.scatter([], [], [], color='royalblue', s=30, label='Tierra')
oumuamua_marker = ax.scatter([], [], [], color='black', s=20, label='Oumuamua')

lim = np.max(np.abs(np.vstack([r_earth_au, r_obj_au])))
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_zlim(-lim, lim)
ax.set_xlabel('X [AU]')
ax.set_ylabel('Y [AU]')
ax.set_zlabel('Z [AU]')
ax.set_title('Oumuamua cerca del Sol (Matplotlib)')
ax.legend(loc='upper right')


def update(frame):
    idx = int(frame)
    earth_line.set_data(r_earth_au[:idx + 1, 0], r_earth_au[:idx + 1, 1])
    earth_line.set_3d_properties(r_earth_au[:idx + 1, 2])

    oumuamua_line.set_data(r_obj_au[:idx + 1, 0], r_obj_au[:idx + 1, 1])
    oumuamua_line.set_3d_properties(r_obj_au[:idx + 1, 2])

    earth_marker._offsets3d = (
        [r_earth_au[idx, 0]],
        [r_earth_au[idx, 1]],
        [r_earth_au[idx, 2]],
    )
    oumuamua_marker._offsets3d = (
        [r_obj_au[idx, 0]],
        [r_obj_au[idx, 1]],
        [r_obj_au[idx, 2]],
    )

    return earth_line, oumuamua_line, earth_marker, oumuamua_marker


anim = FuncAnimation(fig, update, frames=frame_indices, interval=50, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

## Animación 3D con Plotly

In [ ]:
frames = []
for idx in frame_indices:
    frames.append(
        go.Frame(
            data=[
                go.Scatter3d(
                    x=[r_earth_au[idx, 0]],
                    y=[r_earth_au[idx, 1]],
                    z=[r_earth_au[idx, 2]],
                    mode='markers',
                    marker=dict(size=4, color='royalblue'),
                    name='Tierra',
                ),
                go.Scatter3d(
                    x=[r_obj_au[idx, 0]],
                    y=[r_obj_au[idx, 1]],
                    z=[r_obj_au[idx, 2]],
                    mode='markers',
                    marker=dict(size=4, color='black'),
                    name='Oumuamua',
                ),
            ],
            traces=[3, 4],
            name=str(idx),
        )
    )

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=[0], y=[0], z=[0],
            mode='markers',
            marker=dict(size=6, color='gold'),
            name='Sol',
        ),
        go.Scatter3d(
            x=r_earth_au[:, 0],
            y=r_earth_au[:, 1],
            z=r_earth_au[:, 2],
            mode='lines',
            line=dict(color='royalblue', width=2),
            name='Órbita Tierra',
        ),
        go.Scatter3d(
            x=r_obj_au[:, 0],
            y=r_obj_au[:, 1],
            z=r_obj_au[:, 2],
            mode='lines',
            line=dict(color='black', width=2),
            name='Órbita Oumuamua',
        ),
        go.Scatter3d(
            x=[r_earth_au[0, 0]],
            y=[r_earth_au[0, 1]],
            z=[r_earth_au[0, 2]],
            mode='markers',
            marker=dict(size=4, color='royalblue'),
            name='Tierra',
        ),
        go.Scatter3d(
            x=[r_obj_au[0, 0]],
            y=[r_obj_au[0, 1]],
            z=[r_obj_au[0, 2]],
            mode='markers',
            marker=dict(size=4, color='black'),
            name='Oumuamua',
        ),
    ],
    frames=frames,
)

fig.update_layout(
    title='Oumuamua cerca del Sol (Plotly)',
    scene=dict(
        xaxis_title='X [AU]',
        yaxis_title='Y [AU]',
        zaxis_title='Z [AU]',
        aspectmode='data',
    ),
    margin=dict(l=0, r=0, t=40, b=0),
    updatemenus=[
        dict(
            type='buttons',
            showactive=False,
            buttons=[
                dict(label='Play', method='animate', args=[None, {'frame': {'duration': 50, 'redraw': True}}]),
                dict(label='Pause', method='animate', args=[[None], {'frame': {'duration': 0, 'redraw': False}, 'mode': 'immediate'}]),
            ],
        )
    ],
    sliders=[
        dict(
            steps=[
                dict(method='animate', args=[[str(idx)], {'mode': 'immediate', 'frame': {'duration': 0, 'redraw': True}}], label=str(idx))
                for idx in frame_indices
            ],
            currentvalue=dict(prefix='Frame: '),
        )
    ],
)

fig.show()